# STEP 3 — SPEECH GENERATION (Text-to-Speech)

## Useful Information

**Text-to-Speech (TTS)**

Narration Script  
↓  
TTS Model (`gpt-4o-mini-tts`)  
↓  
Narration Audio (.mp3)  
↓  
Combined with Video and Subtitles

Text-to-Speech (TTS) converts the generated narration text into spoken audio. The audio can later be combined with the video and subtitles.

Common TTS options include OpenAI, ElevenLabs, and Google Cloud TTS. This project uses `gpt-4o-mini-tts` because it provides natural speech, different voices, and control over speaking style.

The current price is about **US$0.60 per 1M input text tokens** and **US$12 per 1M audio tokens**. For a short video, the TTS cost is relatively low.

**Traditional TTS**

Text  
↓  
Signal Processing / Pre-recorded Speech Units / Small Speech Models  
↓  
Audio

Traditional TTS systems use:
- signal processing (uses mathematical rules to create and modify speech sounds),

- pre-recorded speech units (a human records many small pieces of speech, then the TTS system joins these pieces together to form sentences),

- or smaller speech models (statistical models predict speech features such as pronunciation and pitch, then a vocoder converts them into audio).

Common tools include:

- **eSpeak / eSpeak NG** – open-source and lightweight TTS.

- **Festival** – an older open-source speech synthesis system.

- **Flite** – a small and fast TTS engine designed for local use.

**Pros:** Can run locally, fast, low computing requirements, and usually free.  

**Cons:** Voices often sound less natural and have limited control over tone and emotion.

**Modern AI-based TTS**

Text  
↓  
Deep Learning Model  
↓  
Natural Speech Audio

Modern TTS uses deep learning and generative AI to produce more natural speech. Common services and tools include:

- **OpenAI TTS** – natural speech with voice and speaking-style control.

- **ElevenLabs** – focuses on realistic and expressive AI voices.

- **Google Cloud Text-to-Speech** – cloud-based TTS with many languages and voices.

- **Coqui TTS** – open-source deep-learning TTS that can run locally.

**Pros:** More natural voices, better expression, and greater control over speaking style.  

**Cons:** Cloud services may have API costs, while local AI models can require more computing resources.

This project uses **OpenAI `gpt-4o-mini-tts`** because it is easy to integrate with Python and provides natural and controllable narration.

## 1. Setup

### 1.1 Import Libraries

In [1]:
# Standard library
import os
import subprocess
from pathlib import Path


# Environment variables
from dotenv import load_dotenv


# Data validation
from pydantic import BaseModel, Field

# OpenAI text-to-speech
from openai import OpenAI

### 1.2 Load Environment Variables

In [2]:
# Load environment variables from .env
# Each provider reads its own key in section 2, so nothing is required yet
load_dotenv()

True

### 1.3 Create Audio Output Directory

In [3]:
# Output directory
OUTPUT_DIR = Path("../outputs/speech_generation")
AUDIO_DIR = OUTPUT_DIR / "audio"

# Create the directory if it does not already exist
AUDIO_DIR.mkdir(parents = True, exist_ok = True)

print(f"Audio directory: {AUDIO_DIR.resolve()}")    # .resolve() converts a Path into its full absolute path.

Audio directory: /Users/yukpinglee/Desktop/agentic-video/outputs/speech_generation/audio


### 1.4 Define Speech Settings

In [4]:
# Style guidance for the narration voice.
# Providers that do not support style instructions ignore this.
SPEECH_INSTRUCTIONS = ("Speak as a clear, warm documentary narrator. "
                       "Use a steady pace and natural pauses between sentences.")

## 2. Configure TTS Provider

### 2.1 Select TTS Provider


In [5]:
TTS_PROVIDER = "openai"
TTS_MODEL = "gpt-4o-mini-tts"

### 2.2 Configure API Credentials

In [6]:
TTS_API_KEY = os.getenv("AI_MODEL_API_KEY")

if not TTS_API_KEY:
    raise ValueError("TTS API key is missing.")

### 2.3 Create TTS Client

In [7]:
def create_tts_client(provider: str, api_key: str):
    """Create a TTS client for the selected provider."""

    if provider == "openai":
        return OpenAI(api_key = api_key)

    elif provider == "elevenlabs":
        return ElevenLabs(api_key = api_key)

    elif provider == "google":
        return GoogleTTSClient(api_key = api_key)

    # This function can be extended to support additional TTS providers in the future.

    raise ValueError(f"Unsupported TTS provider: {provider}")

In [8]:
tts_client = create_tts_client(provider = TTS_PROVIDER, api_key = TTS_API_KEY)

### 2.4 Select Voice and Audio Format

In [9]:
TTS_VOICE = "alloy"
AUDIO_FORMAT = "mp3"

## 3. Speech Models

### 3.1 Define SpeechRequest

In [10]:
class SpeechRequest(BaseModel):
    """Request for generating speech audio."""

    text: str = Field(description = "Text to convert into speech")

    voice: str = Field(default = TTS_VOICE, description = "Voice used for speech generation")

    audio_format: str = Field(default = AUDIO_FORMAT, description = "Audio output format")

### 3.2 Define SpeechResult

In [11]:
class SpeechResult(BaseModel):
    """Result of speech generation."""

    audio_path: str = Field(description = "Path to the generated audio file")

    duration_seconds: float = Field(description = "Duration of the generated audio in seconds")

    provider: str = Field(description = "TTS provider used")

    voice: str = Field(description = "Voice used for speech generation")

## 4. Speech Generation

### 4.1 Generate Speech

In [12]:
# Only supports the OpenAI TTS model at the moment

def generate_speech(request: SpeechRequest) -> bytes:
    """Generate speech using the selected TTS provider."""

    if TTS_PROVIDER == "openai":
        response = tts_client.audio.speech.create(model = TTS_MODEL,
                                                voice = request.voice,
                                                input = request.text,
                                                instructions = SPEECH_INSTRUCTIONS,
                                                response_format = request.audio_format
                                                )

        return response.content

    raise ValueError(f"Unsupported TTS provider: {TTS_PROVIDER}")

### 4.2 Save Generated Audio

In [13]:
def save_audio(audio_bytes: bytes, filename: str) -> Path:
    """Save generated audio to the output directory."""
    # audio_bytes = audio data returned by the TTS API
    output_path = AUDIO_DIR / filename

    output_path.write_bytes(audio_bytes)        # write_bytes() takes binary data and writes it into a file (mp3).

    return output_path

### 4.3 Determine Audio Duration

In [14]:
def get_audio_duration(audio_path: Path) -> float:
    """Return the duration of an audio file in seconds."""

    result = subprocess.run(["ffprobe",
                             "-v",
                             "error",
                             "-show_entries",
                             "format=duration",
                             "-of",
                             "default=noprint_wrappers=1:nokey=1",
                             str(audio_path)],
                            capture_output = True,
                            text = True,
                            check = True
                            )

    return float(result.stdout.strip())

### How `get_audio_duration()` works

- `subprocess.run()` runs the `ffprobe` command-line tool from Python.

- `ffprobe` is part of FFmpeg and is used to inspect media files.

- `"-v", "error"`: only show errors, don't print unnecessary information.

- `"-show_entries", "format=duration"`: only ask for the duration of the audio.

- `"-of", "default=noprint_wrappers=1:nokey=1"`: controls the output format.

  - By default, ffprobe returns:

    ```
    [FORMAT]
    duration=12.540000
    [/FORMAT]
    ```

  - `noprint_wrappers=1` removes the `[FORMAT]` and `[/FORMAT]` lines.

  - `nokey=1` removes the `duration=` label.

  - So the output is simply `12.540000`.

- `str(audio_path)`: converts the `Path` object into a string that ffprobe can use.

- `capture_output = True` and `text = True`: store ffprobe's output as text in `result.stdout` instead of printing it.

- `check = True`: raise an error if ffprobe fails.

- `float(result.stdout.strip())`: removes the trailing newline and converts the text into a number of seconds.

## 5. Generate Narration from VideoScript

### 5.1 Define VideoScript

In [15]:
class VideoScript(BaseModel):
    """Structured narration script for the video (produced in Step 1)."""

    title: str = Field(description = "A short title for the video")
    narration: str = Field(description = "The complete narration script for the video")


example_script = VideoScript(title = "What Are AI Agents?",
                             narration = ("AI agents are systems that can plan, decide, and act on their own. "
                                          "Instead of answering a single question, an agent breaks a goal into steps. "
                                          "It chooses the right tool for each step, checks the result, and keeps going "
                                          "until the goal is reached. "
                                          "That is what makes agents so useful."
                                          ))


### 5.2 Generate Narration

In [16]:
def generate_narration(script: VideoScript) -> SpeechResult:
    """Generate one narration audio file for a VideoScript."""

    request = SpeechRequest(text = script.narration)

    audio_bytes = generate_speech(request)
    audio_path = save_audio(audio_bytes, f"narration.{request.audio_format}")

    # Note: the narration length (duration_seconds) sets the length of the final video (Step 5).
    return SpeechResult(audio_path = str(audio_path),
                        duration_seconds = get_audio_duration(audio_path),
                        provider = TTS_PROVIDER,
                        voice = request.voice
                        )

### 5.3 Test Narration Generation

**PAID** — the cell below calls the selected TTS provider.

In [17]:
# PAID API CALL

speech_result = generate_narration(example_script)

print(f"Saved: {Path(speech_result.audio_path).resolve()}")

Saved: /Users/yukpinglee/Desktop/agentic-video/outputs/speech_generation/audio/narration.mp3
